In [2]:
import duckdb
import pandas as pd
import os

con = duckdb.connect('../data/processed/ecommerce.duckdb')

con.execute("""
    CREATE VIEW IF NOT EXISTS ecommerce AS
    SELECT * FROM read_parquet('../data/processed/ecommerce.parquet')
""")

print("Connection ready")

Connection ready


In [3]:
con.execute("""
    COPY (
        SELECT
            -- Kolom original (bersih)
            event_time,
            event_type,
            product_id,
            category_id,
            price,
            user_id,
            user_session,

            -- Handle nulls
            COALESCE(category_code, 'uncategorized')    AS category_code,
            COALESCE(brand, 'unknown')                  AS brand,

            -- Feature engineering dari timestamp
            DATE(event_time)                            AS event_date,
            EXTRACT(HOUR FROM event_time)               AS hour_of_day,
            EXTRACT(DOW FROM event_time)                AS day_of_week,
            EXTRACT(WEEK FROM event_time)               AS week_number,

            -- Pecah category_code jadi 2 level
            -- contoh: 'electronics.smartphone' → 'electronics'
            SPLIT_PART(
                COALESCE(category_code, 'uncategorized'), '.', 1
            )                                           AS category_l1,
            SPLIT_PART(
                COALESCE(category_code, 'uncategorized'), '.', 2
            )                                           AS category_l2

        FROM ecommerce
        WHERE price > 0          -- buang zero/negative price (68,673 rows)
          AND user_session IS NOT NULL   -- buang 2 null session
    )
    TO '../data/processed/ecommerce_clean.parquet'
    (FORMAT PARQUET, COMPRESSION 'snappy')
""")

print("Cleaning selesai!")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Cleaning selesai!


In [4]:
# Register versi clean
con.execute("""
    CREATE VIEW IF NOT EXISTS ecommerce_clean AS
    SELECT * FROM read_parquet('../data/processed/ecommerce_clean.parquet')
""")

verify = con.execute("""
    SELECT
        COUNT(*)                                AS total_rows,
        COUNT(DISTINCT user_id)                 AS unique_users,
        COUNT(DISTINCT category_l1)             AS unique_category_l1,
        COUNT(DISTINCT brand)                   AS unique_brands,
        ROUND(MIN(price), 2)                    AS min_price,
        ROUND(MAX(price), 2)                    AS max_price,
        ROUND(AVG(price), 2)                    AS avg_price,

        -- Cek null sudah bersih
        COUNT(*) - COUNT(category_code)         AS null_category_code,
        COUNT(*) - COUNT(brand)                 AS null_brand

    FROM ecommerce_clean
""").df()

print("Clean Dataset Overview:")
display(verify.T.rename(columns={0: 'value'}))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Clean Dataset Overview:


,value
total_rows,4.238009e+07
unique_users,3.021435e+06
unique_category_l1,1.400000e+01
unique_brands,3.445000e+03
min_price,7.700000e-01
max_price,2.574070e+03
avg_price,2.907900e+02
null_category_code,0.000000e+00
null_brand,0.000000e+00


In [5]:
preview = con.execute("""
    SELECT
        event_time,
        event_type,
        price,
        category_code,
        category_l1,
        category_l2,
        brand,
        hour_of_day,
        day_of_week
    FROM ecommerce_clean
    LIMIT 10
""").df()

print("Preview cleaned data:")
display(preview)

Preview cleaned data:


,event_time,event_type,price,category_code,category_l1,category_l2,brand,hour_of_day,day_of_week
0,2019-10-01 07:00:00+07:00,view,35.790001,uncategorized,uncategorized,,shiseido,7,2
1,2019-10-01 07:00:00+07:00,view,33.200001,appliances.environment.water_heater,appliances,environment,aqua,7,2
2,2019-10-01 07:00:01+07:00,view,543.099976,furniture.living_room.sofa,furniture,living_room,unknown,7,2
3,2019-10-01 07:00:01+07:00,view,251.740005,computers.notebook,computers,notebook,lenovo,7,2
4,2019-10-01 07:00:04+07:00,view,1081.979980,electronics.smartphone,electronics,smartphone,apple,7,2
5,2019-10-01 07:00:05+07:00,view,908.619995,computers.desktop,computers,desktop,pulser,7,2
6,2019-10-01 07:00:08+07:00,view,380.959991,uncategorized,uncategorized,,creed,7,2
7,2019-10-01 07:00:08+07:00,view,41.160000,uncategorized,uncategorized,,luminarc,7,2
8,2019-10-01 07:00:10+07:00,view,102.709999,apparel.shoes.keds,apparel,shoes,baden,7,2
9,2019-10-01 07:00:11+07:00,view,566.010010,electronics.smartphone,electronics,smartphone,huawei,7,2


In [6]:
removed = 42_448_764 - con.execute(
    "SELECT COUNT(*) FROM ecommerce_clean"
).fetchone()[0]

print("=" * 45)
print("        CLEANING SUMMARY REPORT")
print("=" * 45)
print(f"  Original rows     : 42,448,764")
print(f"  Removed rows      : {removed:,}")
print(f"  └─ zero price     : 68,673")
print(f"  └─ null session   : 2")
print(f"  Clean rows        : {42_448_764 - removed:,}")
print(f"  Data retained     : {((42_448_764-removed)/42_448_764)*100:.2f}%")
print("=" * 45)
print("\n  Null Handling:")
print("  category_code null → 'uncategorized'")
print("  brand null        → 'unknown'")
print("\n  New Features Added:")
print("  event_date")
print("  hour_of_day")
print("  day_of_week")
print("  week_number")
print("  category_l1")
print("  category_l2")
print("=" * 45)

        CLEANING SUMMARY REPORT
  Original rows     : 42,448,764
  Removed rows      : 68,675
  └─ zero price     : 68,673
  └─ null session   : 2
  Clean rows        : 42,380,089
  Data retained     : 99.84%

  Null Handling:
  category_code null → 'uncategorized'
  brand null        → 'unknown'

  New Features Added:
  event_date
  hour_of_day
  day_of_week
  week_number
  category_l1
  category_l2


In [7]:
con.close()
print("Connection closed")

Connection closed
